# MuFu Training

Prepare the data, choose the settings, train MuFu, and inspect the losses.
The baseline runs all five shared seeds (0, 1, 2, 3, 4).


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

AI_DIR = Path.cwd()
if AI_DIR.name == "notebooks":
    AI_DIR = AI_DIR.parent
elif AI_DIR.name != "ai":
    AI_DIR = AI_DIR / "ai"
sys.path.insert(0, str(AI_DIR / "src"))

from baseline.benchmark import SEEDS, load_benchmark_split, run_benchmark
from baseline.results import RESULTS_PATH, summarize_seeds
from encoders.fingerprint_encoder import FingerprintEncoder
from encoders.gnn_encoder import GNNEncoder
from encoders.smiles_encoder import SmilesEncoder
from mufu.data.dataset import split_datasets
from mufu.model import MuFu
from mufu.train import TrainConfig, fit, predict

## Data

Prepare the datasets once, outside the seed loop.


In [ ]:
df = load_benchmark_split()
datasets = split_datasets(df)
for split in datasets:
    print(f"{split}: {len(datasets[split])} molecules")

## Training Settings


In [ ]:
model_name = "mufu"
seeds = SEEDS  # Shared baseline seeds: 0, 1, 2, 3, 4.

num_epochs = 50
batch_size = 64
learn_rate = 0.001
weight_decay = 0.0001
patience = 10
sampler = "mixed"
bio_fraction = 0.5
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## Train

The existing benchmark calls `fit_predict(df, seed)` for each selected seed.
Model construction, training, loss logging, and predictions are shown directly below.


In [ ]:
def fit_predict(df, seed):
    # run_benchmark sets the random seed before calling this function.
    print(f"\nSeed {seed} | device: {device}", flush=True)

    # Model: reuse the existing encoder defaults.
    smiles_encoder = SmilesEncoder(freeze=True, max_cache_size=len(df))
    fingerprint_encoder = FingerprintEncoder()
    graph_encoder = GNNEncoder()
    model = MuFu(smiles_encoder, fingerprint_encoder, graph_encoder)

    # Training: fit already handles batches, losses, early stopping, and saving.
    checkpoint = AI_DIR / "checkpoints" / f"{model_name}_seed_{seed}.pt"
    config = TrainConfig(
        lr=learn_rate,
        weight_decay=weight_decay,
        batch_size=batch_size,
        epochs=num_epochs,
        patience=patience,
        sampler=sampler,
        bio_fraction=bio_fraction,
        num_workers=0,
        device=device,
        checkpoint_path=checkpoint,
    )
    result = fit(model, datasets, config, verbose=True)
    print(f"Best epoch: {result.best_epoch} | val loss: {result.best_val_loss:.4f}")
    print("Checkpoint:", checkpoint)

    # Keep only the losses needed to inspect training and draw the curves.
    history = pd.DataFrame(
        {
            "epoch": range(1, len(result.train_losses) + 1),
            "train_loss": result.train_losses,
            "val_loss": result.val_losses,
        }
    )
    output_dir = AI_DIR / "outputs" / model_name
    output_dir.mkdir(parents=True, exist_ok=True)
    history.to_csv(output_dir / f"seed_{seed}_loss.csv", index=False)

    # fit has restored the best validation-selected model.
    return {
        "val": predict(model, datasets["val"], batch_size=batch_size),
        "test": predict(model, datasets["test"], batch_size=batch_size),
    }

In [ ]:
params = {
    "lr": learn_rate,
    "weight_decay": weight_decay,
    "batch_size": batch_size,
    "epochs": num_epochs,
    "patience": patience,
    "sampler": sampler,
    "bio_fraction": bio_fraction,
    "num_workers": 0,
    "device": device,
}
summary = run_benchmark(
    fit_predict,
    model=model_name,
    encoder="molformer-ecfp4-gine",
    params=params,
    seeds=seeds,
    df=df,
)
columns = ["dataset", "endpoint", "split", "metric", "value", "std", "n_seeds"]
summary[columns].round(4)

## Loss Curves

The log has only epoch, train loss, and validation loss. Read it again without retraining.
Train loss averages batch-level task losses; validation loss uses the full split.


In [ ]:
for seed in seeds:
    history = pd.read_csv(AI_DIR / "outputs" / model_name / f"seed_{seed}_loss.csv")
    plt.figure(figsize=(8, 4))
    plt.plot(history["epoch"], history["train_loss"], label="Training loss")
    plt.plot(history["epoch"], history["val_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model_name} seed {seed}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Recorded Test Results

Use the existing seed summary to compare recorded MuFu and RF results.
`value` is the mean, `std` is sample standard deviation, and `n_seeds` is the number of valid results.
A single seed has undefined std, not zero. This cell only reads results; it does not train.


In [ ]:
results = pd.read_csv(RESULTS_PATH)
comparison = summarize_seeds(results)
comparison = comparison[comparison["split"] == "test"]
columns = ["model", "dataset", "endpoint", "metric", "value", "std", "n_seeds"]
comparison[columns].round(4)

## MuFu vs RF

Test-set mean ± standard deviation across recorded seeds; see `n_seeds` in the table above.
RMSE: lower is better. R² and ROC-AUC: higher is better.
Bioactivity compares the same pIC50 endpoints as RF.


In [ ]:
if model_name not in comparison["model"].values:
    print(
        f"No test results for {model_name} yet. Run training before comparing with RF."
    )
else:
    for rf_name in ["rf_bioactivity", "rf_toxicity"]:
        rf_results = comparison[comparison["model"] == rf_name]
        for metric in rf_results["metric"].unique():
            endpoints = rf_results.loc[rf_results["metric"] == metric, "endpoint"]
            plot_rows = comparison[
                comparison["model"].isin([rf_name, model_name])
                & (comparison["metric"] == metric)
                & comparison["endpoint"].isin(endpoints)
            ]
            means = plot_rows.pivot(index="endpoint", columns="model", values="value")
            errors = plot_rows.pivot(index="endpoint", columns="model", values="std")
            model_order = [rf_name, model_name]
            means = means[model_order]
            errors = errors[model_order]
            ax = means.plot.barh(
                xerr=errors,
                capsize=3,
                figsize=(10, max(4, 0.45 * len(means))),
            )
            direction = "lower is better" if metric == "rmse" else "higher is better"
            ax.set_xlabel(f"{metric} ({direction})")
            ax.set_ylabel("Endpoint")
            ax.set_title(f"{model_name} vs {rf_name} (test)")
            ax.grid(True, axis="x", alpha=0.3)
            plt.tight_layout()
            plt.show()